# 02 · Spend concentration

**Business questions**

- **Q1.** How concentrated is spend across the supplier base?
- **Q1b.** In which categories does the buyer depend on one or a few sellers, and how much spend sits there?

**Framing.** Olist is treated as the buying organization and its sellers as the supplier base. Spend = item price of valid orders (not canceled or unavailable), freight excluded. Window: Jan 2017 to Aug 2018. Concentration uses the Herfindahl-Hirschman Index (HHI, 0 to 10,000) of seller shares within each category; bands follow the 2010 DOJ/FTC guidelines (below 1,500 low, 1,500 to 2,500 moderate, above 2,500 high).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from src.data import q
from src.style import C, MONO, finish, save

pd.set_option("display.float_format", "{:,.2f}".format)

## 1 · Data check
The notebook starts by reconciling with the marts, so every number below traces back to the pipeline.

In [2]:
sellers = q("SELECT seller_id, spend_brl FROM mart_seller_scorecard_base WHERE spend_brl > 0")
cats = q("SELECT * FROM mart_category_concentration")
unknown_spend = q("""SELECT SUM(price) AS s FROM fct_order_items
                     WHERE in_window AND is_valid_status AND category = 'unknown'""")["s"].iloc[0]

sellers["spend_brl"] = sellers["spend_brl"].astype(float)
TOTAL = sellers["spend_brl"].sum()
check = cats["spend_brl"].sum() + float(unknown_spend)

print(f"Sellers with spend:          {len(sellers):,}")
print(f"Total spend (BRL):           {TOTAL:,.0f}")
print(f"Categories + unknown (BRL):  {check:,.0f}   difference {TOTAL - check:,.2f}")
print(f"Categories analysed:         {len(cats)}")

Sellers with spend:          3,029
Total spend (BRL):           13,449,530
Categories + unknown (BRL):  13,449,530   difference 0.00
Categories analysed:         73


## 2 · Pareto of supplier spend (Q1)

In [3]:
p = sellers.sort_values("spend_brl", ascending=False).reset_index(drop=True)
p["rank"] = np.arange(1, len(p) + 1)
p["pct_sellers"] = p["rank"] / len(p)
p["cum_share"] = p["spend_brl"].cumsum() / TOTAL

n80 = int((p["cum_share"] < 0.80).sum() + 1)
pct80 = n80 / len(p)
top10_share = p.loc[p["rank"] <= max(1, round(0.10 * len(p))), "spend_brl"].sum() / TOTAL
tail_share = 1 - p.loc[p["rank"] <= n80, "spend_brl"].sum() / TOTAL

print(f"{n80:,} sellers ({pct80:.1%}) account for 80% of spend")
print(f"Top 10% of sellers hold {top10_share:.1%} of spend")
print(f"The remaining {len(p) - n80:,} sellers ({1 - pct80:.1%}) share {tail_share:.1%} of spend")

537 sellers (17.7%) account for 80% of spend
Top 10% of sellers hold 67.4% of spend
The remaining 2,492 sellers (82.3%) share 20.0% of spend


In [4]:
fig = go.Figure()
fig.add_scatter(x=p["pct_sellers"], y=p["cum_share"], mode="lines",
                line=dict(color=C["forest_700"], width=2.5), name="Cumulative spend share",
                hovertemplate="Top %{x:.1%} of sellers<br>%{y:.1%} of spend<extra></extra>")
# reference: perfectly even spend
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", name="Even distribution",
                line=dict(color=C["context"], width=1.5, dash="dot"), hoverinfo="skip")
# key point: square data node (design system device), forest on light ground
fig.add_scatter(x=[pct80], y=[0.80], mode="markers", showlegend=False, hoverinfo="skip",
                marker=dict(symbol="square", size=11, color=C["forest_900"]))
fig.add_shape(type="line", x0=pct80, x1=pct80, y0=0, y1=0.80,
              line=dict(color=C["baseline"], width=1, dash="dash"))
fig.add_shape(type="line", x0=0, x1=pct80, y0=0.80, y1=0.80,
              line=dict(color=C["baseline"], width=1, dash="dash"))
fig.add_annotation(x=pct80, y=0.80, ax=60, ay=40, showarrow=True, arrowcolor=C["axis"],
                   arrowwidth=1, xanchor="left",
                   text=f"<b>{n80:,} sellers</b> · {pct80:.1%} of the base<br>hold 80% of spend",
                   font=dict(size=12, color=C["ink"]))
fig.update_xaxes(title="Share of sellers, ranked by spend", tickformat=".0%", range=[0, 1])
fig.update_yaxes(title="Cumulative share of spend", tickformat=".0%", range=[0, 1.02])
finish(fig, f"{pct80:.0%} of sellers account for 80% of spend",
       f"Cumulative spend share by seller rank · {len(p):,} sellers · BRL {TOTAL/1e6:,.1f}M item spend")
fig.show()
save(fig, "01_pareto_spend")

WindowsPath('C:/dev/procurement-analytics-olist/reports/figures/01_pareto_spend.png')

## 3 · Category concentration (Q1b)
A high HHI matters only where the category carries money. Categories are classed as **material** when they hold at least 1% of total spend; dependency risk is flagged when a material category has high concentration or a single seller above 50% of its spend.

In [5]:
MATERIAL = 0.01

c = cats.copy()
for col in ["spend_brl", "hhi", "top1_share_pct"]:
    c[col] = c[col].astype(float)
c["spend_share"] = c["spend_brl"] / TOTAL
c["material"] = c["spend_share"] >= MATERIAL
c["at_risk"] = c["material"] & ((c["hhi"] > 2500) | c["single_source_risk"])
c["label"] = c["category"].str.replace("_", " ")

weighted_hhi = (c["hhi"] * c["spend_brl"]).sum() / c["spend_brl"].sum()
share = lambda mask: c.loc[mask, "spend_brl"].sum() / TOTAL

print(f"Spend-weighted HHI across categories:          {weighted_hhi:,.0f}")
print(f"Material categories (>= 1% of spend):           {c['material'].sum()} of {len(c)}, "
      f"{share(c['material']):.1%} of spend")
print(f"Spend in high-concentration categories (any):   {share(c['hhi'] > 2500):.1%}")
print(f"Spend in single-source-risk categories (any):   {share(c['single_source_risk']):.1%}")
print(f"Material categories at risk:                    {c['at_risk'].sum()}, {share(c['at_risk']):.1%} of spend")

Spend-weighted HHI across categories:          788
Material categories (>= 1% of spend):           25 of 73, 89.6% of spend
Spend in high-concentration categories (any):   5.1%
Spend in single-source-risk categories (any):   4.2%
Material categories at risk:                    2, 3.7% of spend


In [6]:
fig = go.Figure()
groups = [
    ("Not material (< 1% of spend)", ~c["material"], C["context"], 8),
    ("Material, low or moderate concentration", c["material"] & ~c["at_risk"], C["forest_700"], 11),
    ("Material, dependency risk", c["at_risk"], C["unfavorable"], 12),
]
for name, mask, color, size in groups:
    d = c[mask]
    fig.add_scatter(x=d["spend_share"], y=d["hhi"], mode="markers", name=name,
                    marker=dict(symbol="square", size=size, color=color),
                    customdata=np.stack([d["label"], d["n_sellers"], d["top1_share_pct"]], axis=-1),
                    hovertemplate="<b>%{customdata[0]}</b><br>%{x:.1%} of spend · HHI %{y:,.0f}"
                                  "<br>%{customdata[1]} sellers · top seller %{customdata[2]:.0f}%<extra></extra>")

for level, txt in [(1500, "MODERATE · 1,500"), (2500, "HIGH · 2,500")]:
    fig.add_hline(y=level, line=dict(color=C["baseline"], width=1, dash="dash"))
    fig.add_annotation(x=1, xref="paper", y=level, yanchor="bottom", xanchor="right",
                       text=txt, showarrow=False, font=dict(family=MONO, size=10, color=C["axis"]))
fig.add_vline(x=MATERIAL, line=dict(color=C["baseline"], width=1, dash="dot"))

# direct labels only where action is needed
for _, r in c[c["at_risk"]].iterrows():
    fig.add_annotation(x=np.log10(r["spend_share"]), y=r["hhi"], text=r["label"], showarrow=False,
                       xanchor="left", xshift=9, font=dict(size=11, color=C["unfavorable"]))

ticks = [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1]
fig.update_xaxes(type="log", title="Category share of total spend (log scale)",
                 tickvals=ticks, ticktext=["0.01%", "0.03%", "0.1%", "0.3%", "1%", "3%", "10%"])
fig.update_yaxes(title="HHI of seller shares within category", range=[0, 10500], tickformat=",.0f")
n_risk = int(c["at_risk"].sum())
title = (f"Only {n_risk} material categories depend on few sellers, {share(c['at_risk']):.1%} of spend"
         if n_risk else "No material category depends on few sellers")
finish(fig, title, "Seller concentration (HHI) vs category spend share · material = at least 1% of spend")
fig.show()
save(fig, "02_category_concentration")

WindowsPath('C:/dev/procurement-analytics-olist/reports/figures/02_category_concentration.png')

### Sensitivity of the materiality cut
The 1% threshold is a judgement call. If the at-risk list changes a lot between 0.5% and 2%, the conclusion depends on the threshold and must say so.

In [7]:
rows = []
for t in [0.005, 0.01, 0.02]:
    m = c["spend_share"] >= t
    risk = m & ((c["hhi"] > 2500) | c["single_source_risk"])
    rows.append({"threshold": f"{t:.1%}", "material_categories": int(m.sum()),
                 "at_risk": int(risk.sum()), "at_risk_spend": f"{share(risk):.1%}",
                 "categories": ", ".join(c.loc[risk, "category"])})
pd.DataFrame(rows)

,threshold,material_categories,at_risk,at_risk_spend,categories
0,0.5%,30,2,3.7%,"office_furniture, computers"
1,1.0%,25,2,3.7%,"office_furniture, computers"
2,2.0%,15,1,2.0%,office_furniture


### Categories ranked for action

In [8]:
table = (c[c["material"]]
         .sort_values(["at_risk", "hhi"], ascending=[False, False])
         [["category", "spend_share", "n_sellers", "top1_share_pct", "hhi", "concentration_level", "at_risk"]]
         .rename(columns={"spend_share": "share_of_spend", "top1_share_pct": "top_seller_pct"}))
table.style.format({"share_of_spend": "{:.1%}", "top_seller_pct": "{:.0f}%", "hhi": "{:,.0f}"}).hide(axis="index")

category,share_of_spend,n_sellers,top_seller_pct,hhi,concentration_level,at_risk
computers,1.7%,9,75%,"5,796",high,True
office_furniture,2.0%,34,64%,"4,284",high,True
stationery,1.7%,173,35%,"1,342",low,False
watches_gifts,8.9%,98,17%,931,low,False
luggage_accessories,1.0%,73,22%,921,low,False
musical_instruments,1.4%,68,20%,913,low,False
consoles_games,1.1%,81,24%,863,low,False
perfumery,2.9%,172,17%,808,low,False
garden_tools,3.6%,235,21%,723,low,False
small_appliances,1.4%,101,19%,680,low,False


## 4 · Findings for `docs/findings.md`
Generated from the numbers above, so the text never drifts from the data. Review the wording before copying.

In [9]:
risk_names = ", ".join(c.loc[c["at_risk"], "label"].head(3)) or "none"
hi_share = share(c["hhi"] > 2500)
n_hi = int((c["hhi"] > 2500).sum())

findings = [
    f"**{pct80:.0%} of sellers hold 80% of spend.** {n80:,} of {len(p):,} sellers account for 80% of "
    f"BRL {TOTAL/1e6:,.1f}M item spend (valid orders, Jan 2017 to Aug 2018).",
]
if weighted_hhi < 1500:
    findings.append(
        f"**Concentration is a category problem, not a portfolio problem.** Spend-weighted HHI is "
        f"{weighted_hhi:,.0f} (low), but {int(c['at_risk'].sum())} material categories "
        f"({share(c['at_risk']):.1%} of spend) show high concentration or a single seller above 50%: {risk_names}.")
else:
    findings.append(
        f"**Supplier dependency is widespread.** Spend-weighted HHI is {weighted_hhi:,.0f}; "
        f"{int(c['at_risk'].sum())} material categories ({share(c['at_risk']):.1%} of spend) are at risk: {risk_names}.")
if hi_share < 0.20:
    findings.append(
        f"**Most high-HHI categories are immaterial.** {n_hi} categories have high HHI but hold only "
        f"{hi_share:.1%} of spend: an HHI ranking without a spend filter would point effort at the wrong places.")
else:
    findings.append(
        f"**High concentration carries real money.** {n_hi} high-HHI categories hold {hi_share:.1%} of spend.")

for f in findings:
    print("- " + f + "\n")
print("Implication to write by hand: what a category manager should do with each finding.")

- **18% of sellers hold 80% of spend.** 537 of 3,029 sellers account for 80% of BRL 13.4M item spend (valid orders, Jan 2017 to Aug 2018).

- **Concentration is a category problem, not a portfolio problem.** Spend-weighted HHI is 788 (low), but 2 material categories (3.7% of spend) show high concentration or a single seller above 50%: office furniture, computers.

- **Most high-HHI categories are immaterial.** 20 categories have high HHI but hold only 5.1% of spend: an HHI ranking without a spend filter would point effort at the wrong places.

Implication to write by hand: what a category manager should do with each finding.
